In [1]:
import requests
import json
import pandas as pd
import time

# ตั้งค่า URL ของ API (เปลี่ยนเป็น http://localhost:8000/api/v1 ถ้าไม่ได้รันใน Docker Network เดียวกัน)
BASE_URL = 'http://api:8000/api/v1'

# 1. โหลด Golden Data (เฉลย) จากไฟล์ CSV
df_test = pd.read_csv('./input/testcase - ชีต1.csv')

# Dictionary สำรองสำหรับ file_path ในกรณีที่ในไฟล์ CSV ยังไม่ได้เพิ่มคอลัมน์ 'file_path'
fallback_paths = {
    "Sirichat Thueman": "./input/_Resume Sirichat - sirichat thueman.pdf",
    "Kitichai Paichayon": "./input/kitichai - kitichai paichayon.pdf",
    "Phonlachet Tepanwong": "./input/Phonlachet_Resume_112025 - Phonlachet Tepanwong.pdf",
    "Bantoon Suksangkaew": "./input/Resume_Bantoon - บัณฑูรย์ สุขแสงแก้ว.pdf",
    "Kritsada Wuthanusorn": "./input/Resume_Kritsada Wuthanusorn_Digitalstoremesh - กฤษฎา วุฒานุสรณ์.pdf",
    "Nanthida Claisuban": "./input/resume_nathida - Pop Nanthida.pdf",
    "Supakarn Manakitphaisan": "./input/Supakarn Manakitphaisan_Resume - Supakarn Prink Manakitphaisan.pdf",
    "Nutcharee Yodnirod": "./input/เอกสารสมัครงาน - นุชรีย์ ยอดนิโรด.pdf",
    "Anan Wanna": "./input/Mr.Anan.pdf",
    "Tanawat Sangwong": "./input/sample2.pdf",
    "Poovanut Sritareerit": "./input/sample3.pdf"
}

# แปลง DataFrame ให้กลายเป็น List of Dictionaries
test_cases = []
for _, row in df_test.iterrows():
    name = str(row['name']).strip() if pd.notna(row.get('name')) else ""
    
    # ดึง file_path จาก CSV (ถ้ามี) หรือใช้ Fallback path
    f_path = row.get('file_path')
    if pd.isna(f_path) or not f_path:
        f_path = fallback_paths.get(name, "")

    test_cases.append({
        "file_path": f_path,
        "name": name,
        "target_jobs": [job.strip() for job in str(row['classifier_job']).split(',')] if pd.notna(row.get('classifier_job')) else [],
        "golden_skills": [skill.strip() for skill in str(row['skill']).split(',')] if pd.notna(row.get('skill')) else []
    })

def clean_strings(str_list):
    """แปลงตัวอักษรเป็นพิมพ์เล็กและตัดช่องว่าง เพื่อให้เทียบคำกันได้แม่นยำ"""
    return [str(s).strip().lower() for s in str_list]

# เก็บผลลัพธ์
evaluation_results = []

print("🚀 กำลังเริ่มประเมินความแม่นยำ AI (Evaluating...)")
print("-" * 50)

for idx, tc in enumerate(test_cases):
    print(f"[{idx+1}/{len(test_cases)}] กำลังวิเคราะห์: {tc['name']}")
    
    # ข้ามถ้าไม่มีไฟล์ PDF
    if not tc["file_path"]:
        print(f"  ❌ ไม่พบพาธไฟล์ PDF สำหรับ {tc['name']}")
        continue
        
    # --- 1. ยิง API: สกัดข้อมูล (Extract) ---
    ai_extracted_skills = []
    try:
        with open(tc["file_path"], 'rb') as f:
            res_ext = requests.post(f"{BASE_URL}/extract", files={'file': f})
            res_ext.raise_for_status()
            ai_extracted_skills = res_ext.json().get('data', {}).get('skill', [])
    except Exception as e:
        print(f"  ❌ Error สกัดข้อมูลไฟล์ {tc['file_path']}: {e}")
        continue

    # --- เตรียมเปรียบเทียบคำ (Metrics) ---
    golden_original = tc["golden_skills"]
    golden_clean_map = {str(s).strip().lower(): str(s).strip() for s in golden_original}
    ai_skills_clean = set(clean_strings(ai_extracted_skills))
    
    matched_skills = [golden_clean_map[s] for s in golden_clean_map.keys() if s in ai_skills_clean]
    missed_skills = [golden_clean_map[s] for s in golden_clean_map.keys() if s not in ai_skills_clean]

    # คำนวณ Skill Recall & Precision
    true_positives = len(matched_skills)
    total_golden = len(golden_clean_map)
    recall_pct = (true_positives / total_golden) * 100 if total_golden > 0 else 0
    precision_pct = (true_positives / len(ai_skills_clean)) * 100 if len(ai_skills_clean) > 0 else 0

    # --- 2. ยิง API: จับคู่อาชีพ (Match) ---
    ai_matched_jobs = []
    job_match_acc = None  # ตั้งค่าเริ่มต้นเป็น None
    job_status = ""

    # เช็คว่ามีสกิลที่ดึงได้หรือไม่ ถ้าไม่มีให้ "ข้าม" การยิง API จับคู่ไปเลย (ป้องกันเปอร์เซ็นต์ล่วง)
    if not ai_extracted_skills:
        print(f"  ⚠️ สกัดทักษะไม่ได้ (0 Skills) -> ข้ามการจับคู่อาชีพเพื่อไม่ให้ฉุดคะแนน")
        job_status = "Skipped (No Skills)"
    else:
        try:
            res_match = requests.post(f"{BASE_URL}/match", json={'skills': ai_extracted_skills})
            res_match.raise_for_status()
            ai_matched_jobs = [job['job_name'] for job in res_match.json().get('data', {}).get('top_jobs', [])]
        except Exception as e:
            print(f"  ❌ Error จับคู่อาชีพ {tc['name']}: {e}")
        
        # คำนวณ Job Match Accuracy
        target_jobs_clean = clean_strings(tc["target_jobs"])
        ai_jobs_clean = clean_strings(ai_matched_jobs)
        
        is_expected_none = any("none" in job for job in target_jobs_clean)
        
        if is_expected_none:
            job_match_acc = 100.0 if len(ai_jobs_clean) == 0 else 0.0
            job_status = "Pass (None)" if job_match_acc == 100 else "Fail (Expected None)"
        else:
            is_matched = any(job in ai_jobs_clean for job in target_jobs_clean)
            job_match_acc = 100.0 if is_matched else 0.0
            job_status = "Pass" if is_matched else "Fail"

    # เก็บผลลัพธ์ลงตาราง
    evaluation_results.append({
        "Name": tc["name"],
        "Recall (%)": round(recall_pct, 2),
        "Precision (%)": round(precision_pct, 2),
        "Matched Skills (ดึงได้)": ", ".join(matched_skills) if matched_skills else "-",
        "Missed Skills (หาไม่เจอ)": ", ".join(missed_skills) if missed_skills else "-",
        "Job Match Acc (%)": job_match_acc, # ค่าจะเป็น None ถ้าถูกข้าม
        "Expected Jobs": ", ".join(tc["target_jobs"]),
        "AI Predicted Jobs": ", ".join(ai_matched_jobs) if ai_matched_jobs else "-",
        "Status": job_status
    })
    
    # หน่วงเวลาเล็กน้อยเพื่อไม่ให้ API ทำงานหนักเกินไป
    time.sleep(1)

# --- 4. สรุปผลลัพธ์เป็น DataFrame ---
df_results = pd.DataFrame(evaluation_results)

print("\n" + "=" * 50)
print("🎯 สรุปผลการทดสอบความแม่นยำ (Accuracy Report)")
print("=" * 50)

# แสดงตารางรายบุคคล
display(df_results) # ใช้ print(df_results) ถ้าไม่ได้รันใน Jupyter

# คำนวณค่าเฉลี่ยรวมของระบบ (dropna() จะช่วยตัดแถวที่ค่าเป็น None ทิ้งไปก่อนหาค่าเฉลี่ย ทำให้เปอร์เซ็นต์ไม่ตก)
if len(df_results) > 0:
    overall_recall = df_results["Recall (%)"].dropna().mean()
    overall_precision = df_results["Precision (%)"].dropna().mean()
    
    # เช็คว่ามีข้อมูล Job Match ให้คำนวณไหม
    valid_job_match = df_results["Job Match Acc (%)"].dropna()
    overall_job_acc = valid_job_match.mean() if len(valid_job_match) > 0 else 0.0

    print("\n📊 ภาพรวมทั้งระบบ (System Average):")
    print(f"  - Skill Extraction (Recall):    {overall_recall:.2f}% (ความครบถ้วนของการดึงข้อมูล)")
    print(f"  - Skill Extraction (Precision): {overall_precision:.2f}% (ความแม่นยำ ไม่ดึงศัพท์มั่ว)")
    print(f"  - Job Matching Accuracy:        {overall_job_acc:.2f}% (อาชีพเฉลยติด Top 3 - ไม่นับไฟล์ที่ Error)")
else:
    print("\n⚠️ ไม่มีผลลัพธ์ให้แสดง กรุณาตรวจสอบไฟล์ PDF หรือ CSV")

🚀 กำลังเริ่มประเมินความแม่นยำ AI (Evaluating...)
--------------------------------------------------
[1/12] กำลังวิเคราะห์: Sasipreeya Phanthaburee
  ❌ ไม่พบพาธไฟล์ PDF สำหรับ Sasipreeya Phanthaburee
[2/12] กำลังวิเคราะห์: Kitichai Paichayon
[3/12] กำลังวิเคราะห์: Phonlachet Tepanwong
[4/12] กำลังวิเคราะห์: Bantoon Suksangkaew
[5/12] กำลังวิเคราะห์: Kritsada Wuthanusorn
[6/12] กำลังวิเคราะห์: Nanthida Claisuban
[7/12] กำลังวิเคราะห์: Supakarn Manakitphaisan
[8/12] กำลังวิเคราะห์: Nutcharee Yodnirod
  ⚠️ สกัดทักษะไม่ได้ (0 Skills) -> ข้ามการจับคู่อาชีพเพื่อไม่ให้ฉุดคะแนน
[9/12] กำลังวิเคราะห์: Anan Wanna
[10/12] กำลังวิเคราะห์: Waraporn Wongbut
  ❌ ไม่พบพาธไฟล์ PDF สำหรับ Waraporn Wongbut
[11/12] กำลังวิเคราะห์: Tanawat Sangwong
  ⚠️ สกัดทักษะไม่ได้ (0 Skills) -> ข้ามการจับคู่อาชีพเพื่อไม่ให้ฉุดคะแนน
[12/12] กำลังวิเคราะห์: Poovanut Sritareerit
  ⚠️ สกัดทักษะไม่ได้ (0 Skills) -> ข้ามการจับคู่อาชีพเพื่อไม่ให้ฉุดคะแนน

🎯 สรุปผลการทดสอบความแม่นยำ (Accuracy Report)


,Name,Recall (%),Precision (%),Matched Skills (ดึงได้),Missed Skills (หาไม่เจอ),Job Match Acc (%),Expected Jobs,AI Predicted Jobs,Status
0,Kitichai Paichayon,41.67,15.15,"Node.js, HTML, Javascript, Python, Docker","API, SQL, Web Development, Backend Development...",100.0,Software Developer and Prompt,"Software Developer and Prompt , Cybersecurity ...",Pass
1,Phonlachet Tepanwong,37.50,27.27,"Cybersecurity, UX/UI, Data Analytics","Agile, Scrum Framework, Incident Response, Net...",100.0,Cybersecurity Specialist,"Cybersecurity Specialist, Data Analyst, UXUI",Pass
2,Bantoon Suksangkaew,50.00,15.79,"Python, HTML, SQL","Problem Solving, Communication Skills, Error R...",100.0,Software Developer and Prompt,"Software Developer and Prompt , Cybersecurity ...",Pass
3,Kritsada Wuthanusorn,50.00,18.75,"SQL, Javascript, Node.js, UiPath, Docker, ChatGPT","Data Analytics, Machine Learning, API Developm...",100.0,"Software Developer and Prompt, Data Engineer","Software Developer and Prompt , Database Admin...",Pass
4,Nanthida Claisuban,63.64,31.82,"HTML, Javascript, SQL, Python, CSS, Figma, Sof...","UX/UI, UI Design, Web Development, Frontend De...",100.0,"Software Developer and Prompt, UXUI","Software Developer and Prompt , UXUI, Data Ana...",Pass
5,Supakarn Manakitphaisan,30.00,21.43,"Power BI, SQL, Python","Data Integration, ETL Process, ETL/ELT, Data E...",100.0,"Data Engineer, Data Analyst","Data Analyst, Software Developer and Prompt , ...",Pass
6,Nutcharee Yodnirod,0.00,0.00,-,"Excel, Invoice Management, Inventory Managemen...",NaN,None (ไม่มีตำแหน่งที่ตรงในระบบ),-,Skipped (No Skills)
7,Anan Wanna,50.00,18.18,"Excel, Leadership","Presentation Skills, Data Analysis",0.0,None (ไม่มีตำแหน่งที่ตรงในระบบ),"Digital Transformation, Data Analyst, Business...",Fail (Expected None)
8,Tanawat Sangwong,0.00,0.00,-,"Python, HTML, CSS, Javascript, SQL, UX/UI, Fro...",NaN,Software Developer and Prompt,-,Skipped (No Skills)
9,Poovanut Sritareerit,0.00,0.00,-,"Python, SQL, Javascript, Git Command, Version ...",NaN,Software Developer and Prompt,-,Skipped (No Skills)



📊 ภาพรวมทั้งระบบ (System Average):
  - Skill Extraction (Recall):    32.28% (ความครบถ้วนของการดึงข้อมูล)
  - Skill Extraction (Precision): 14.84% (ความแม่นยำ ไม่ดึงศัพท์มั่ว)
  - Job Matching Accuracy:        85.71% (อาชีพเฉลยติด Top 3 - ไม่นับไฟล์ที่ Error)


In [3]:
df_results.to_csv('./eva.csv', index=False)